# Author Analysis v2

Cross-community author trajectory analysis for r/fednews, r/jobs, and r/layoffs.

## Step 1: Build combined author list with first post date

For each author across all three subreddits, find their **earliest** post date. If an author posted in multiple source subreddits, keep only the earliest one. Output ordered: fednews → jobs → layoffs.

In [2]:
# Step 1: Build a deduplicated author list with each author's earliest post date.
# Reads per-subreddit author CSVs, resolves duplicates (authors in multiple source subs)
# by keeping the earliest date, and assigns each author to that source subreddit.

import csv
from collections import defaultdict

BASE = '../subredditwise_author_lists_csv'

# Processing order matters: fednews → jobs (all batches) → layoffs
FILES = [
    (f'{BASE}/authors_of_fednews_subreddit.csv', 'fednews'),
    (f'{BASE}/authors_of_jobs_subreddit.csv', 'jobs'),
    (f'{BASE}/authors_of_jobs_subreddit_batch2.csv', 'jobs'),
    (f'{BASE}/authors_of_jobs_subreddit_batch3.csv', 'jobs'),
    (f'{BASE}/authors_of_jobs_subreddit_batch4.csv', 'jobs'),
    (f'{BASE}/authors_of_layoffs_subreddit.csv', 'layoffs'),
]

# Track earliest (date, subreddit) per author across all source files
author_earliest = {}

for fpath, subreddit in FILES:
    with open(fpath, 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            author = row['author'].strip()
            date = row['date'].strip()
            if not author or not date:
                continue
            # Keep whichever source subreddit has the earlier date
            if author not in author_earliest or date < author_earliest[author][0]:
                author_earliest[author] = (date, subreddit)

# Group authors by their assigned source subreddit
groups = {'fednews': [], 'jobs': [], 'layoffs': []}
for author, (date, sub) in author_earliest.items():
    groups[sub].append((author, date, sub))

# Write output ordered: fednews first, then jobs, then layoffs
OUTPUT = 'all_authors_first_post.csv'
total = 0
with open(OUTPUT, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['author', 'first_post_date', 'subreddit'])
    for sub in ['fednews', 'jobs', 'layoffs']:
        for row in groups[sub]:
            w.writerow(row)
            total += 1

print(f'Written {total} unique authors to {OUTPUT}')
for sub in ['fednews', 'jobs', 'layoffs']:
    print(f'  {sub}: {len(groups[sub])}')

Written 206778 unique authors to all_authors_first_post.csv
  fednews: 23164
  jobs: 178615
  layoffs: 4999


## Step 2: Extract per-author subreddit timelines

Read the raw JSONL files (~6.9GB, 208K authors) and extract `(date, subreddit)` pairs for every post. Output: one JSONL line per author with their full posting timeline, sorted by date. Duplicates across raw files are skipped.

**Configurable filters** (set to 0 = no filtering by default):
- `MIN_TOTAL_POSTS`: minimum total posts across all subreddits
- `MIN_OTHER_SUBS`: minimum number of distinct subreddits posted in *besides* the source subreddit

In [3]:
# Step 2: Extract per-author posting timelines from raw data (~6.9GB).
# For each author, pull every (date, subreddit) pair across all of Reddit,
# producing a chronological timeline. Skips duplicates across raw files.

import json
import csv
from pathlib import Path

# ====== CONFIGURABLE FILTERS (change these as needed) ======
MIN_TOTAL_POSTS = 0    # minimum total posts across all subreddits (0 = no filter)
MIN_OTHER_SUBS = 0     # minimum distinct subreddits besides source subreddit (0 = no filter)
# ============================================================

# --- Load author metadata from Step 1 output ---
author_meta = {}  # author -> (first_post_date, source_subreddit)
with open('all_authors_first_post.csv', 'r') as f:
    for row in csv.DictReader(f):
        author_meta[row['author']] = (row['first_post_date'], row['subreddit'])

print(f"Loaded metadata for {len(author_meta)} authors")
print(f"Filters: MIN_TOTAL_POSTS={MIN_TOTAL_POSTS}, MIN_OTHER_SUBS={MIN_OTHER_SUBS}")

# --- Process raw JSONL files (one per source subreddit) ---
RAW_DIR = Path('../author_activity_across_subreddits')
RAW_FILES = [
    'fednews_author_data.jsonl',
    'jobs_author_data_batch1.jsonl',
    'jobs_author_data_batch2.jsonl',
    'jobs_author_data_batch3.jsonl',
    'jobs_author_data_batch4.jsonl',
    'layoffs_author_data.jsonl',
]

OUTPUT = 'all_authors_subreddit_timeline.jsonl'
seen = set()  # deduplicate authors appearing in multiple raw files
total_written = 0
total_posts = 0
skipped_dup = 0
skipped_no_meta = 0
skipped_min_posts = 0
skipped_min_subs = 0

with open(OUTPUT, 'w') as out:
    for fname in RAW_FILES:
        fpath = RAW_DIR / fname
        file_written = 0
        file_dup = 0
        
        with open(fpath, 'r') as f:
            for line in f:
                record = json.loads(line)
                author = record.get('user', {}).get('author', '')
                
                # Skip if already processed from an earlier file
                if not author or author in seen:
                    file_dup += 1
                    continue
                seen.add(author)
                
                if author not in author_meta:
                    skipped_no_meta += 1
                    continue
                
                first_post_date, source_sub = author_meta[author]
                
                # Extract (date, subreddit) pairs from all posts
                timeline = []
                for post in record.get('posts', []):
                    date = post.get('date')
                    subreddit = post.get('subreddit')
                    if date and subreddit:
                        timeline.append({'date': date, 'subreddit': subreddit})
                
                # --- Apply configurable filters ---
                if MIN_TOTAL_POSTS > 0 and len(timeline) < MIN_TOTAL_POSTS:
                    skipped_min_posts += 1
                    continue
                
                if MIN_OTHER_SUBS > 0:
                    other_subs = {t['subreddit'] for t in timeline} - {source_sub}
                    if len(other_subs) < MIN_OTHER_SUBS:
                        skipped_min_subs += 1
                        continue
                
                # Sort chronologically for trajectory analysis
                timeline.sort(key=lambda x: x['date'])
                
                out_record = {
                    'author': author,
                    'source_subreddit': source_sub,
                    'first_post_date': first_post_date,
                    'timeline': timeline,
                    'total_posts': len(timeline)
                }
                out.write(json.dumps(out_record) + '\n')
                file_written += 1
                total_posts += len(timeline)
        
        total_written += file_written
        skipped_dup += file_dup
        print(f"{fname}: {file_written} authors written, {file_dup} skipped (duplicate)")

print(f"\n--- Summary ---")
print(f"Total authors written: {total_written}")
print(f"Total posts in timelines: {total_posts}")
print(f"Skipped (duplicate across files): {skipped_dup}")
print(f"Skipped (no metadata in CSV): {skipped_no_meta}")
print(f"Skipped (below MIN_TOTAL_POSTS={MIN_TOTAL_POSTS}): {skipped_min_posts}")
print(f"Skipped (below MIN_OTHER_SUBS={MIN_OTHER_SUBS}): {skipped_min_subs}")
print(f"Authors in CSV but not in raw files: {len(author_meta) - total_written - skipped_min_posts - skipped_min_subs}")

Loaded metadata for 206778 authors
Filters: MIN_TOTAL_POSTS=0, MIN_OTHER_SUBS=0
fednews_author_data.jsonl: 23429 authors written, 27 skipped (duplicate)
jobs_author_data_batch1.jsonl: 44271 authors written, 5729 skipped (duplicate)
jobs_author_data_batch2.jsonl: 47269 authors written, 2731 skipped (duplicate)
jobs_author_data_batch3.jsonl: 48186 authors written, 1814 skipped (duplicate)
jobs_author_data_batch4.jsonl: 28084 authors written, 885 skipped (duplicate)
layoffs_author_data.jsonl: 4560 authors written, 1200 skipped (duplicate)

--- Summary ---
Total authors written: 195799
Total posts in timelines: 11725008
Skipped (duplicate across files): 12386
Skipped (no metadata in CSV): 0
Skipped (below MIN_TOTAL_POSTS=0): 0
Skipped (below MIN_OTHER_SUBS=0): 0
Authors in CSV but not in raw files: 10979


## Step 3: Exploratory distribution analysis

Understand the shape of the data before building trajectory features. This informs filter thresholds and whether the anchor event definition works.

In [4]:
# 3a: Load all author timelines and count authors per source subreddit.

import json
import numpy as np
from collections import Counter

authors = []
with open('all_authors_subreddit_timeline.jsonl', 'r') as f:
    for line in f:
        authors.append(json.loads(line))

print(f"Loaded {len(authors)} authors")
src_counts = Counter(a['source_subreddit'] for a in authors)
for sub in ['fednews', 'jobs', 'layoffs']:
    print(f"  {sub}: {src_counts[sub]}")

Loaded 195799 authors
  fednews: 23137
  jobs: 167823
  layoffs: 4839


In [5]:
# 3b: Post count distribution — how active are authors overall?
# Broken down by percentiles and by source subreddit to understand
# where to set minimum activity thresholds.

post_counts = [a['total_posts'] for a in authors]

print("=== Post count distribution (all authors) ===")
percentiles = [0, 10, 25, 50, 75, 90, 95, 99, 100]
vals = np.percentile(post_counts, percentiles)
for p, v in zip(percentiles, vals):
    print(f"  P{p:3d}: {v:.0f}")
print(f"  Mean: {np.mean(post_counts):.1f}")
print(f"  Authors with 0 posts: {sum(1 for c in post_counts if c == 0)}")
print(f"  Authors with 1 post:  {sum(1 for c in post_counts if c == 1)}")
print(f"  Authors with ≤5 posts: {sum(1 for c in post_counts if c <= 5)}")

# fednews authors are notably sparser (median=3 vs 16-18 for others)
print("\n=== Post count by source subreddit (median / mean) ===")
for sub in ['fednews', 'jobs', 'layoffs']:
    counts = [a['total_posts'] for a in authors if a['source_subreddit'] == sub]
    print(f"  {sub:8s}: median={np.median(counts):.0f}, mean={np.mean(counts):.1f}, n={len(counts)}")

=== Post count distribution (all authors) ===
  P  0: 1
  P 10: 2
  P 25: 4
  P 50: 14
  P 75: 44
  P 90: 111
  P 95: 191
  P 99: 555
  P100: 1087486
  Mean: 59.9
  Authors with 0 posts: 0
  Authors with 1 post:  16882
  Authors with ≤5 posts: 56816

=== Post count by source subreddit (median / mean) ===
  fednews : median=3, mean=12.1, n=23137
  jobs    : median=18, mean=66.3, n=167823
  layoffs : median=16, mean=64.4, n=4839


In [6]:
# 3c: Subreddit diversity — how many distinct communities does each author post in?
# "Other-sub diversity" excludes the source subreddit to measure cross-community reach.
# Key for deciding if an author has enough signal for trajectory analysis.

diversity = []
other_sub_counts = []
only_source = 0

for a in authors:
    subs = {t['subreddit'] for t in a['timeline']}
    diversity.append(len(subs))
    # Count subs excluding the one they were recruited from
    other = subs - {a['source_subreddit']}
    other_sub_counts.append(len(other))
    if len(other) == 0:
        only_source += 1

print("=== Subreddit diversity (distinct subs per author) ===")
percentiles = [0, 10, 25, 50, 75, 90, 95, 99, 100]
vals = np.percentile(diversity, percentiles)
for p, v in zip(percentiles, vals):
    print(f"  P{p:3d}: {v:.0f}")

print(f"\n=== Other-subreddit diversity (excluding source sub) ===")
vals = np.percentile(other_sub_counts, percentiles)
for p, v in zip(percentiles, vals):
    print(f"  P{p:3d}: {v:.0f}")
print(f"\n  Authors posting ONLY in source subreddit: {only_source} ({only_source/len(authors)*100:.1f}%)")
print(f"  Authors with ≥1 other sub: {len(authors) - only_source}")
print(f"  Authors with ≥3 other subs: {sum(1 for c in other_sub_counts if c >= 3)}")
print(f"  Authors with ≥5 other subs: {sum(1 for c in other_sub_counts if c >= 5)}")

# Note: 37% of fednews authors never post outside fednews — limits trajectory analysis for gov sector
print(f"\n=== Only-source-sub authors by source ===")
for sub in ['fednews', 'jobs', 'layoffs']:
    sub_authors = [a for a in authors if a['source_subreddit'] == sub]
    sub_only = sum(1 for a in sub_authors if len({t['subreddit'] for t in a['timeline']} - {sub}) == 0)
    print(f"  {sub:8s}: {sub_only}/{len(sub_authors)} ({sub_only/len(sub_authors)*100:.1f}%) post only in source")

=== Subreddit diversity (distinct subs per author) ===
  P  0: 1
  P 10: 1
  P 25: 3
  P 50: 9
  P 75: 22
  P 90: 45
  P 95: 67
  P 99: 141
  P100: 4415

=== Other-subreddit diversity (excluding source sub) ===
  P  0: 0
  P 10: 0
  P 25: 2
  P 50: 8
  P 75: 21
  P 90: 44
  P 95: 66
  P 99: 140
  P100: 4414

  Authors posting ONLY in source subreddit: 20209 (10.3%)
  Authors with ≥1 other sub: 175590
  Authors with ≥3 other subs: 146063
  Authors with ≥5 other subs: 125140

=== Only-source-sub authors by source ===
  fednews : 8590/23137 (37.1%) post only in source
  jobs    : 11619/167823 (6.9%) post only in source
  layoffs : 0/4839 (0.0%) post only in source


In [7]:
# 3d: Before/after anchor event analysis.
# Anchor = first post date in source subreddit. Tests how many authors
# have posting activity on BOTH sides of the anchor — required for
# meaningful before-vs-after trajectory comparison.
# Key finding: fednews only 28.4% have both, vs ~65% for jobs/layoffs.

has_before = 0
has_after = 0
has_both = 0
no_timeline = 0

before_counts = []
after_counts = []

for a in authors:
    anchor = a['first_post_date']
    timeline = a['timeline']
    
    if not timeline:
        no_timeline += 1
        continue
    
    # Count posts strictly before and after the anchor date
    b = sum(1 for t in timeline if t['date'] < anchor)
    af = sum(1 for t in timeline if t['date'] > anchor)
    before_counts.append(b)
    after_counts.append(af)
    
    if b > 0:
        has_before += 1
    if af > 0:
        has_after += 1
    if b > 0 and af > 0:
        has_both += 1

print("=== Before/after anchor event (first post in source subreddit) ===")
print(f"  Total authors with timelines: {len(authors) - no_timeline}")
print(f"  Authors with no timeline: {no_timeline}")
print(f"  Has posts BEFORE anchor: {has_before} ({has_before/len(authors)*100:.1f}%)")
print(f"  Has posts AFTER anchor:  {has_after} ({has_after/len(authors)*100:.1f}%)")
print(f"  Has BOTH before & after: {has_both} ({has_both/len(authors)*100:.1f}%)")
print(f"  Has NEITHER (only on anchor date): {len(authors) - no_timeline - has_before - has_after + has_both}")

print(f"\n=== Before-anchor post count distribution (authors with ≥1 before) ===")
before_nonzero = [c for c in before_counts if c > 0]
if before_nonzero:
    percentiles = [0, 25, 50, 75, 90, 95, 100]
    vals = np.percentile(before_nonzero, percentiles)
    for p, v in zip(percentiles, vals):
        print(f"  P{p:3d}: {v:.0f}")

print(f"\n=== After-anchor post count distribution (authors with ≥1 after) ===")
after_nonzero = [c for c in after_counts if c > 0]
if after_nonzero:
    vals = np.percentile(after_nonzero, percentiles)
    for p, v in zip(percentiles, vals):
        print(f"  P{p:3d}: {v:.0f}")

# Breakdown by source — shows fednews is the outlier
print(f"\n=== Has-both-before-and-after by source ===")
for sub in ['fednews', 'jobs', 'layoffs']:
    sub_authors = [a for a in authors if a['source_subreddit'] == sub]
    both = 0
    for a in sub_authors:
        anchor = a['first_post_date']
        b = any(t['date'] < anchor for t in a['timeline'])
        af = any(t['date'] > anchor for t in a['timeline'])
        if b and af:
            both += 1
    print(f"  {sub:8s}: {both}/{len(sub_authors)} ({both/len(sub_authors)*100:.1f}%) have both before & after")

=== Before/after anchor event (first post in source subreddit) ===
  Total authors with timelines: 195799
  Authors with no timeline: 0
  Has posts BEFORE anchor: 140230 (71.6%)
  Has posts AFTER anchor:  149204 (76.2%)
  Has BOTH before & after: 120452 (61.5%)
  Has NEITHER (only on anchor date): 26817

=== Before-anchor post count distribution (authors with ≥1 before) ===
  P  0: 1
  P 25: 3
  P 50: 9
  P 75: 25
  P 90: 65
  P 95: 112
  P100: 88012

=== After-anchor post count distribution (authors with ≥1 after) ===
  P  0: 1
  P 25: 3
  P 50: 9
  P 75: 29
  P 90: 75
  P 95: 134
  P100: 1087473

=== Has-both-before-and-after by source ===
  fednews : 6578/23137 (28.4%) have both before & after
  jobs    : 110737/167823 (66.0%) have both before & after
  layoffs : 3137/4839 (64.8%) have both before & after
